# HW2 — Part 1: REINFORCE with Baseline
### Variance Reduction · 2-Armed Bandit · Policy Gradient Analysis
**Yogeshvar Reddy Kallam** · IST 597 Deep RL · Penn State Spring 2025

---

## Problem: REINFORCE Gradient Variance Analysis

**Bandit:** 2 actions — left (reward=2), right (reward=1)  
**Policy:** π_θ(left)=1/(1+e^θ), π_θ(right)=e^θ/(1+e^θ)  
**Initial θ=0** → both arms equally likely (0.5 each)

**Key result:** baseline b=E[R]=1.5 reduces gradient variance from **0.5625 → 0**

In [ ]:
import numpy as np, torch, torch.optim as optim, matplotlib.pyplot as plt

# ── Analytical computation ────────────────────────────────────────────────
theta = 0.0
pi_L, pi_R = 0.5, 0.5
r_L, r_R   = 2.0, 1.0

# Log-policy gradients at θ=0
g_L = r_L * (-pi_R)   # = -1.0
g_R = r_R * ( pi_L)   # = +0.5

E_g   = pi_L*g_L + pi_R*g_R
Var_g = pi_L*g_L**2 + pi_R*g_R**2 - E_g**2

print(f"WITHOUT baseline:  E[∇J] = {E_g:.4f}   Var[∇J] = {Var_g:.4f}")

b    = 0.5*r_L + 0.5*r_R   # 1.5
gb_L = (r_L - b) * (-pi_R)
gb_R = (r_R - b) * ( pi_L)

E_gb   = pi_L*gb_L + pi_R*gb_R
Var_gb = pi_L*gb_L**2 + pi_R*gb_R**2 - E_gb**2

print(f"WITH baseline b={b}: E[∇J] = {E_gb:.4f}   Var[∇J] = {Var_gb:.4f}")
print("✅ Same expected gradient, zero variance!")


In [ ]:
# ── Simulation: convergence with and without baseline ─────────────────────
def run_reinforce(use_baseline, n=5000, lr=0.1, seed=0):
    torch.manual_seed(seed)
    theta = torch.tensor([0.0], requires_grad=True)
    opt   = optim.SGD([theta], lr=lr)
    hist  = []
    for _ in range(n):
        pi_r = torch.sigmoid(theta)
        probs = torch.stack([1-pi_r, pi_r]).squeeze()
        a = torch.multinomial(probs, 1).item()
        r = [2.0, 1.0][a]
        lp  = torch.log(probs[a])
        adv = (r - 1.5) if use_baseline else r
        (-lp * adv).backward(); opt.step(); opt.zero_grad()
        hist.append(theta.item())
    return hist

h0 = run_reinforce(False)
h1 = run_reinforce(True)

plt.figure(figsize=(10,4))
plt.plot(h0, label="No baseline",     alpha=0.7)
plt.plot(h1, label="Baseline b=1.5",  alpha=0.7)
plt.xlabel("Step"); plt.ylabel("θ")
plt.title("REINFORCE convergence (θ→negative = prefers left arm, reward=2)")
plt.legend(); plt.tight_layout(); plt.show()
